# Check evidence packets

Change `packet_id`, then restart the kernel and run all cells.
Run the setup cell first; later cells use its variables.
Nothing here changes files or calls a model.

Sources are extracted from PDFs in `private/pdfs/` into `.txt` and `.json` files in `private/sources/`. The JSON records the PDF filename, hashes, extraction details, and page spans. Apple uses `SRC_01`; Microsoft uses `SRC_02`.

In [6]:
import json
from pathlib import Path
from decimal import Decimal
from pprint import pprint

# Find the repo even when Jupyter starts in src/ or a notebook folder.
repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/benchmark/private/packets").is_dir()), None)
if repo is None:
    raise FileNotFoundError("Open this notebook from inside the agent_test repository.")
folder = repo / "data/benchmark"
packet_id = "EP_0004"  # EP_0001 = Apple, EP_0002 = Microsoft

packet = json.loads((folder / f"private/packets/{packet_id}.json").read_text(encoding="utf-8"))
masked_packet = json.loads((folder / f"public/evidence_packets/{packet_id}.json").read_text(encoding="utf-8"))
source_id = packet["source_id"]
source = (folder / f"private/sources/{source_id}.txt").read_text(encoding="utf-8")
source_info = json.loads((folder / f"private/sources/{source_id}.json").read_text(encoding="utf-8"))

print("Company:", packet["company"]["name"])
print("Original source:", source_info.get("input_file") or source_info.get("source_url"))
print("Saved text:", folder / f"private/sources/{source_id}.txt")

Company: Bank of America Corporation
Original source: private/pdfs/bac-20251231.pdf
Saved text: /Users/claudia/Desktop/dsa4213/dsa4213 proj/DSA4213_Group11/data/benchmark/private/sources/SRC_04.txt


## The evidence packet

`private` has the original names and source references. `public` has the masked version the agents will see.

In [7]:
pprint(packet, sort_dicts=False)

{'schema_version': '0.1',
 'packet_id': 'EP_0004',
 'company_alias': 'COMPANY_04',
 'company': {'name': 'Bank of America Corporation', 'ticker': 'BAC'},
 'fiscal_year': 2025,
 'filing': {'form': '10-K',
            'accession_number': '0000070858-26-000157',
            'filing_date': '2026-02-25',
            'source_url': 'https://www.sec.gov/Archives/edgar/data/70858/000007085826000157/bac-20251231.htm'},
 'source_id': 'SRC_04',
 'evidence': [{'evidence_id': 'EP_0004_E01',
               'section': 'Item 7',
               'text': 'Table 1 Summary Income Statement and Selected '
                       'Financial Data\n'
                       '(Dollars in millions, except per share information) '
                       '2025 2024\n'
                       'Income statement\n'
                       'Net interest income $ 60,096 $ 56,060 \n'
                       'Noninterest income 53,001 49,796 \n'
                       'Total revenue, net of interest expense 113,097 '
          

## Source text vs stored evidence

`source_span` is `[start, end]` in the saved source text. The end character is excluded.
Each comparison should print `Equal: True`. Use the printed locator to check the original filing too.

In [8]:
for evidence in packet["evidence"]:
    start, end = evidence["source_span"]
    source_excerpt = source[start:end]

    print("\n---", evidence["evidence_id"], "---")
    print("Location:", evidence["locator"])
    print("Source span:", [start, end])
    print("\nFROM SOURCE:\n", source_excerpt)
    print("\nIN PACKET:\n", evidence["text"])
    print("\nEqual:", source_excerpt == evidence["text"])


--- EP_0004_E01 ---
Location: PDF p. 29; Printed p. 27; Financial Highlights, Table 1, complete table and explanatory notes; 2025 and revised 2024 columns
Source span: [181266, 182802]

FROM SOURCE:
 Table 1 Summary Income Statement and Selected Financial Data
(Dollars in millions, except per share information) 2025 2024
Income statement
Net interest income $ 60,096 $ 56,060 
Noninterest income 53,001 49,796 
Total revenue, net of interest expense 113,097 105,856 
Provision for credit losses 5,675 5,821 
Noninterest expense 69,727 66,812 
Income before income taxes 37,695 33,223 
Income tax expense 7,186 6,250 
Net income 30,509 26,973 
Preferred stock dividends and other 1,454 1,629 
Net income applicable to common shareholders $ 29,055 $ 25,344 
Per common share information   
Earnings $ 3.86 $ 3.23 
Diluted earnings 3.81 3.19 
Dividends paid 1.08 1.00 
Performance ratios
Return on average assets 0.89 % 0.82 %
Return on average common shareholders’ equity 10.59 9.53 
Return on avera

## Source numbers vs stored numbers

Find each recorded source number in the original excerpt and compare it with the stored value.
Also check the printed context: a matching number alone does not prove it belongs to the right row, year, or unit.

In [9]:
for fact in packet["facts"]:
    evidence = next(e for e in packet["evidence"] if e["evidence_id"] == fact["evidence_id"])
    start, end = evidence["source_span"]
    source_excerpt = source[start:end]
    position = source_excerpt.find(fact["source_value"])
    source_number = source_excerpt[position:position + len(fact["source_value"])] if position >= 0 else None

    print("\n---", fact["fact_id"], fact["metric"], "---")
    print("Evidence:", fact["evidence_id"], "|", fact["locator"])
    print("Period:", fact["period"], "| Unit:", fact["unit"])
    if position >= 0:
        print("Source context:\n", source_excerpt[max(0, position - 100):position + len(fact["source_value"]) + 100])
    print("Source number:", source_number)
    print("Packet number:", fact["value"])
    equal = source_number is not None and Decimal(source_number.replace(",", "")) == Decimal(fact["value"])
    print("Equal:", equal)


--- EP_0004_F01 total_revenue_net_of_interest_expense ---
Evidence: EP_0004_E01 | Table 1; Total revenue, net of interest expense; 2025 column
Period: {'label': 'FY2025', 'start': '2025-01-01', 'end': '2025-12-31'} | Unit: USD_million
Source context:
  income $ 60,096 $ 56,060 
Noninterest income 53,001 49,796 
Total revenue, net of interest expense 113,097 105,856 
Provision for credit losses 5,675 5,821 
Noninterest expense 69,727 66,812 
Income before 
Source number: 113,097
Packet number: 113097
Equal: True

--- EP_0004_F02 total_revenue_net_of_interest_expense ---
Evidence: EP_0004_E01 | Table 1; Total revenue, net of interest expense; 2024 column (revised)
Period: {'label': 'FY2024', 'start': '2024-01-01', 'end': '2024-12-31'} | Unit: USD_million
Source context:
 $ 60,096 $ 56,060 
Noninterest income 53,001 49,796 
Total revenue, net of interest expense 113,097 105,856 
Provision for credit losses 5,675 5,821 
Noninterest expense 69,727 66,812 
Income before income t
Source numb

## Original vs masked

Read each pair. Names and brands should change; the numbers and meaning should stay the same.

In [10]:
for evidence in packet["evidence"]:
    masked = next(e for e in masked_packet["evidence"] if e["evidence_id"] == evidence["evidence_id"])
    print("\n---", evidence["evidence_id"], "---")
    print("ORIGINAL:\n", evidence["text"])
    print("\nMASKED:\n", masked["text"])

print("\nStored numbers unchanged by masking:")
for fact in packet["facts"]:
    masked_fact = next(f for f in masked_packet["facts"] if f["fact_id"] == fact["fact_id"])
    print(fact["fact_id"], fact["value"], "->", masked_fact["value"], "Equal:", fact["value"] == masked_fact["value"])


--- EP_0004_E01 ---
ORIGINAL:
 Table 1 Summary Income Statement and Selected Financial Data
(Dollars in millions, except per share information) 2025 2024
Income statement
Net interest income $ 60,096 $ 56,060 
Noninterest income 53,001 49,796 
Total revenue, net of interest expense 113,097 105,856 
Provision for credit losses 5,675 5,821 
Noninterest expense 69,727 66,812 
Income before income taxes 37,695 33,223 
Income tax expense 7,186 6,250 
Net income 30,509 26,973 
Preferred stock dividends and other 1,454 1,629 
Net income applicable to common shareholders $ 29,055 $ 25,344 
Per common share information   
Earnings $ 3.86 $ 3.23 
Diluted earnings 3.81 3.19 
Dividends paid 1.08 1.00 
Performance ratios
Return on average assets 0.89 % 0.82 %
Return on average common shareholders’ equity 10.59 9.53 
Return on average tangible common shareholders’
equity 14.22 12.94 
Efficiency ratio 61.65 63.12 
Balance sheet at year end   
Total loans and leases $ 1,185,700 $ 1,095,835 
Total ass

## Notes

Keep any problems found here in the matching report review notebook,
for example `private/reviews/EP_0001_review.ipynb`.
That one notebook holds the evidence checks, report feedback, edits and final output.